### Import Libraries

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import  transforms
from torchvision import  datasets
from torch.utils.data import DataLoader

from sklearn.metrics import accuracy_score
import numpy as np

### Device

In [2]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


### Data Augmentation

In [3]:
train_transform=transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

In [4]:
test_transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

### Load Dataset

In [5]:
train_datset=datasets.CIFAR10(root="./data",train=True,download=True,transform=train_transform)
test_dataset=datasets.CIFAR10(root="./data",train=False,download=True,transform=test_transform)

100%|██████████| 170M/170M [45:42<00:00, 62.2kB/s]


### DataLoader

In [6]:
train_loader= DataLoader(train_datset,batch_size=32,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=32,shuffle=False)

### CNN Architecture

In [14]:
class Cifar10CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1=nn.Conv2d(3,32,3)
        self.bn1=nn.BatchNorm2d(32)
        self.conv2=nn.Conv2d(32,64,3)
        self.bn2=nn.BatchNorm2d(64)
        self.conv3=nn.Conv2d(64,128,3)
        self.bn3=nn.BatchNorm2d(128)
        self.pool=nn.MaxPool2d(2,2)
        self.dropout=nn.Dropout(0.25)
        self.fc1=nn.Linear(128*2*2,512)
        self.fc2=nn.Linear(512,10)

    def forward(self,x):
        x=self.conv1(x)
        x=self.bn1(x)
        x=torch.relu(x)
        x=self.pool(x)
        x=self.dropout(x)
        x=self.conv2(x)
        x=self.bn2(x)
        x=torch.relu(x)
        x=self.pool(x)
        x=self.dropout(x)
        x=self.conv3(x)
        x=self.bn3(x)
        x=torch.relu(x)
        x=self.pool(x)
        x=self.dropout(x)
        x=torch.flatten(x,start_dim=1)
        x=torch.relu(self.fc1(x))
        x=self.dropout(x)
        x=self.fc2(x)

        return x

### Create Model

In [15]:
model=Cifar10CNN().to(device)

### Loss Function

In [16]:
criterion=nn.CrossEntropyLoss()

### Optimizer

In [17]:
optimizer=optim.Adam(model.parameters(),lr=0.001)

### Training Loop

In [18]:
epochs=10
for epoch in range(epochs):
    model.train()
    train_loss=0.0
    for images,labels in train_loader:
        images,labels=images.to(device),labels.to(device)
        optimizer.zero_grad()
        outputs=model(images)
        loss=criterion(outputs,labels)
        loss.backward()
        optimizer.step()
        train_loss+=loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_loss/len(train_loader)}")

Epoch 1/10, Train Loss: 1.4592967937561616
Epoch 2/10, Train Loss: 1.215664473696542
Epoch 3/10, Train Loss: 1.1134113507704024
Epoch 4/10, Train Loss: 1.0560154834772941
Epoch 5/10, Train Loss: 1.0073879224058153
Epoch 6/10, Train Loss: 0.9766985125177119
Epoch 7/10, Train Loss: 0.9411228716144635
Epoch 8/10, Train Loss: 0.9199075195092233
Epoch 9/10, Train Loss: 0.8943074627023283
Epoch 10/10, Train Loss: 0.8777000188102954


### Evaluation

In [19]:
model.eval()
correct=0
total=0

with torch.no_grad():
    for images,labels in test_loader:
        images,labels=images.to(device),labels.to(device)
        outputs=model(images)
        _,predicted=torch.max(outputs.data,1)
        total+=labels.size(0)
        correct+=(predicted==labels).sum().item()



### Accuracy

In [23]:
accuracy=(correct/total)*100
print(f"Accuracy: {accuracy}%")

Accuracy: 74.8%


### Save Model

In [21]:
torch.save(model.state_dict(),"cifar10_model.pth")

### Load Model

In [22]:
loaded_model=Cifar10CNN()
loaded_model.load_state_dict(torch.load("cifar10_model.pth"))
loaded_model.eval()

Cifar10CNN(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1))
  (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
  (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1))
  (bn3): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (dropout): Dropout(p=0.25, inplace=False)
  (fc1): Linear(in_features=512, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=10, bias=True)
)